##  Level 2. Data Preparation
**Objective:** Handle missing values, fix inconsistent data, engineer new features, and export a clean dataset for analysis.

In [36]:

# 1. Import Libraries & Load Dat

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [5]:
df = pd.read_csv('Big_Mart_Sales.csv')
print(f' dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')

 dataset loaded: 5681 rows × 11 columns


In [35]:
### 2. Fix Inconsistent Values — Item_Fat_Content ('LF', 'low fat' should be 'Low Fat' , 'reg' should be 'Regular')

print('Before standardization:')
print(df['Item_Fat_Content'].value_counts())

fat_map = {
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
}
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_map)

print('\nAfter standardization:')
print(df['Item_Fat_Content'].value_counts())
print(' Item_Fat_Content standardized!')

Before standardization:
Item_Fat_Content
Low Fat    3668
Regular    2013
Name: count, dtype: int64

After standardization:
Item_Fat_Content
Low Fat    3668
Regular    2013
Name: count, dtype: int64
 Item_Fat_Content standardized!


In [34]:
### 3.Handle Missing Values — Item_Weight (Fill with median weight per Item_Type)

print(f'Missing Item_Weight before: {df["Item_Weight"].isnull().sum()}')

df['Item_Weight'] = df.groupby('Item_Type')['Item_Weight'].transform(
    lambda x: x.fillna(x.median())
)

print(f'Missing Item_Weight after: {df["Item_Weight"].isnull().sum()}')
print('Item_Weight missing values filled with per-category median!')

Missing Item_Weight before: 0
Missing Item_Weight after: 0
Item_Weight missing values filled with per-category median!


In [33]:
### 4. Handle Missing Values — Outlet_Size ( Fill with mode)

def fill_size(group):
    mode_val = group['Outlet_Size'].mode()
    if len(mode_val) > 0:
        group['Outlet_Size'] = group['Outlet_Size'].fillna(mode_val[0])
    return group

df = df.groupby('Outlet_Type', group_keys=False).apply(fill_size)
df['Outlet_Size'] = df['Outlet_Size'].fillna('Small')

print(f"Missing Outlet_Size: {df['Outlet_Size'].isnull().sum()}")
print(' Outlet_Size missing values filled!')

Missing Outlet_Size: 0
 Outlet_Size missing values filled!


In [32]:
### 5. Feature Engineering  ( Outlet Age)

# Create Outlet_Age column
df['Outlet_Age'] = 2025 - df['Outlet_Establishment_Year']

print(' Outlet_Age column created!')
print(df[['Outlet_Identifier', 'Outlet_Establishment_Year', 'Outlet_Age']].drop_duplicates().sort_values('Outlet_Age', ascending=False))

 Outlet_Age column created!
   Outlet_Identifier  Outlet_Establishment_Year  Outlet_Age
4             OUT027                       1985          40
12            OUT019                       1985          40
14            OUT013                       1987          38
5             OUT046                       1997          28
2             OUT010                       1998          27
0             OUT049                       1999          26
8             OUT045                       2002          23
21            OUT035                       2004          21
1             OUT017                       2007          18
6             OUT018                       2009          16


In [31]:
### 6. Format & Verify Numerical and Categorical Fields

# Round float columns to 2 decimal places
df['Item_Weight'] = df['Item_Weight'].round(2)
df['Item_MRP'] = df['Item_MRP'].round(2)
df['Item_Visibility'] = df['Item_Visibility'].round(4)

# Categorical columns — check all look clean
cat_cols = ['Item_Fat_Content', 'Item_Type', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']
for col in cat_cols:
    df[col] = df[col].str.strip()  # remove any leading/trailing spaces

print(' Numerical fields rounded, categorical fields stripped!')

 Numerical fields rounded, categorical fields stripped!


In [30]:
### 7.Final Check (Clean Dataset)
print("Missing values after cleaning:")
print(df.isnull().sum())
print(f"\nFinal shape: {df.shape}")

Missing values after cleaning:
Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Outlet_Age                   0
dtype: int64

Final shape: (5681, 12)


In [37]:
# Preview clean dataset
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Outlet_Age
0,FDW58,20.75,Low Fat,0.0076,Snack Foods,107.86,OUT049,1999,Medium,Tier 1,Supermarket Type1,26
1,FDW14,8.30,Regular,0.0384,Dairy,87.32,OUT017,2007,Small,Tier 2,Supermarket Type1,18
2,NCN55,14.60,Low Fat,0.0996,Others,241.75,OUT010,1998,Small,Tier 3,Grocery Store,27
3,FDQ58,7.32,Low Fat,0.0154,Snack Foods,155.03,OUT017,2007,Small,Tier 2,Supermarket Type1,18
4,FDY38,12.80,Regular,0.1186,Dairy,234.23,OUT027,1985,Medium,Tier 3,Supermarket Type3,40


In [39]:
### 8.Export Clean Dataset

df.to_csv('Big_Mart_Sales_Cleaned.csv', index=False)
print('Clean dataset saved as: Big_Mart_Sales_Cleaned.csv')
print(f'   Rows: {df.shape[0]} | Columns: {df.shape[1]}')

Clean dataset saved as: Big_Mart_Sales_Cleaned.csv
   Rows: 5681 | Columns: 12
